In [ ]:
%%bash
set -eo pipefail

root="$PWD"
work="$root/archives/cryo-benchmark"
mkdir -p "$work"

if [ "$(uname -s)" = Linux ] && ! dpkg -s libgl1 >/dev/null 2>&1; then
    apt-get update
    apt-get install -y libgl1
fi

export PATH="$work/bin:$PATH"
if ! command -v uv >/dev/null; then
    curl -fsSL https://astral.sh/uv/install.sh | UV_UNMANAGED_INSTALL="$work/bin" sh
fi
cd "$work"

repo="$work/MICA"
if [ ! -d "$repo" ]; then
    git clone https://github.com/jianlin-cheng/MICA.git "$repo"
fi
git -C "$repo" checkout --quiet --detach 726d1bb

env="$work/MICA-env"
index=(--index-url https://download.pytorch.org/whl/cu124)
open3d=open3d-cpu==0.19.0
if [ "$(uname -s)" = Darwin ]; then index=(); open3d=open3d==0.19.0; fi
if [ ! -x "$env/bin/python" ]; then uv venv --python 3.11 "$env"; fi
uv pip install --python "$env/bin/python" "${index[@]}" torch==2.5.1
uv pip install --python "$env/bin/python" "$open3d" numpy==1.26.4 scipy==1.13.1 \
    biopython==1.83 superpose3d==1.1.1 networkx==3.4.2 rotary-embedding-torch==0.8.9 \
    mrcfile psutil tqdm einops natsort matplotlib requests


In [ ]:
%%bash
set -eo pipefail
 
root="$PWD"
work="$root/archives/cryo-benchmark"

phenix_env="${PHENIX_ENV:-$work/phenix/phenix_env.sh}"
if [ ! -f "$phenix_env" ]; then
    for script in "$root/../cryo-benchmark-work/phenix/phenix_env.sh" "$HOME"/phenix-*/phenix_env.sh /Applications/phenix-*/phenix_env.sh; do
        if [ -f "$script" ]; then phenix_env="$script"; break; fi
    done
fi

if [ ! -f "$phenix_env" ]; then
    platform="Linux-x86_64"
    if [ "$(uname -s)" = Darwin ]; then platform="MacOSX-arm64"; fi
    for installer in "$root"/Phenix-*"$platform".sh "$HOME"/Downloads/Phenix-*"$platform".sh; do
        if [ -f "$installer" ]; then
            bash "$installer" -b -p "$work/phenix"
            break
        fi
    done
fi
if [ ! -f "$phenix_env" ]; then
    echo "Phenix not found"
    exit 1
fi

method=MICA
repo="$work/$method"
export PATH="$work/MICA-env/bin:$PATH" MPLBACKEND=Agg
device=cuda:0
if [ "$(uname -s)" = Darwin ]; then device=cpu; fi
cd "$repo"

for target in 22XP 32MC; do
    data="$root/data/targets/$target"
    map=("$data"/inputs/emd_*.map)
    fasta="$data/inputs/rcsb_pdb_${target}.fasta"
    out="$work/runs/$method/$target"

    af3="$out/AF3_results"
    mkdir -p "$af3/job"
    cp "$data"/predictions/af3/*/*_model_0.cif "$af3/job/"
    resolution=3.20
    if [ "$target" = 32MC ]; then resolution=2.95; fi

    python -u utils/process_AF3_results.py -f "$fasta" -a "$af3" -d "$device"
    time python -u utils/dock_in_map.py -m "${map[0]}" -f "$fasta" -a "$af3" \
        -c 0.06 -r "$resolution" -x "$phenix_env"
done


In [ ]:
%%bash
set -eo pipefail

root="$PWD"
work="$root/archives/cryo-benchmark"

phenix_env="${PHENIX_ENV:-$work/phenix/phenix_env.sh}"
if [ ! -f "$phenix_env" ]; then
    for script in "$root/../cryo-benchmark-work/phenix/phenix_env.sh" "$HOME"/phenix-*/phenix_env.sh /Applications/phenix-*/phenix_env.sh; do
        if [ -f "$script" ]; then phenix_env="$script"; break; fi
    done
fi

method=MICA
repo="$work/$method"
export PATH="$work/MICA-env/bin:$PATH" MPLBACKEND=Agg
cd "$repo"
if [ ! -s trained_models/MICA_best_model.pth ]; then
    curl -fL 'https://zenodo.org/records/15756654/files/trained_models.tar.gz?download=1' | tar -xz
fi
device=cuda:0
pulchra="$repo/modules/pulchra304/src/pulchra"
if [ "$(uname -s)" = Darwin ]; then
    device=cpu
    pulchra="$repo/pulchra-arm64"
    if [ ! -x "$pulchra" ]; then
        (cd modules/pulchra304/src && cc -O3 -include time.h pulchra.c pulchra_data.c -lm -o "$pulchra")
    fi
    
    python - <<'PYCODE'
from pathlib import Path
path = Path("utils/modeler.py")
path.write_text(path.read_text().replace(
    "from multiprocessing import Pool", "from multiprocessing.pool import ThreadPool as Pool"
))
PYCODE
fi

for target in 22XP 32MC; do
    data="$root/data/targets/$target"
    map=("$data"/inputs/emd_*.map)
    fasta="$data/inputs/rcsb_pdb_${target}.fasta"
    out="$work/runs/$method/$target"

    resolution=3.20
    if [ "$target" = 32MC ]; then resolution=2.95; fi
    time python -u run.py -m "${map[0]}" -f "$fasta" -a "$out/AF3_results" \
        -r "$resolution" --device "$device" --run_pulchra -p "$pulchra" \
        --no_parallel --mul_proc_num 4 -o "$out"

    model=("$out"/*_MICA_all_atom_model.pdb)
    (cd "$out" && source "$phenix_env" && phenix.real_space_refine "${model[0]}" \
        "${map[0]}" "$repo/modules/phenix.eff" "resolution=$resolution")
done


In [ ]:
%%bash
set -eo pipefail

root="$PWD"
work="$root/archives/cryo-benchmark"

if [ "$(uname -s)" = Linux ] && ! command -v g++ >/dev/null; then
    apt-get update
    apt-get install -y g++
fi

phenix_env="${PHENIX_ENV:-$work/phenix/phenix_env.sh}"
if [ ! -f "$phenix_env" ]; then
    for script in "$root/../cryo-benchmark-work/phenix/phenix_env.sh" "$HOME"/phenix-*/phenix_env.sh /Applications/phenix-*/phenix_env.sh; do
        if [ -f "$script" ]; then phenix_env="$script"; break; fi
    done
fi

method="MICA"
if [ ! -d "$work/USalign" ]; then
    git clone https://github.com/pylelab/USalign.git "$work/USalign"
fi
make -C "$work/USalign" USalign

for target in 22XP 32MC; do
    out="$work/runs/$method/$target"
    reference="$root/data/targets/$target/reference/$target.cif"
    model="$(find "$out" -maxdepth 1 \( -name '*_real_space_refined*.pdb' -o -name '*_real_space_refined*.cif' \) -print -quit)" 
    test -s "$model"
    "$work/USalign/USalign" "$model" "$reference" -mol prot -mm 1 -ter 1 -outfmt 2 > "$out/usalign.tsv"
    if [ -f "$phenix_env" ]; then
        (source "$phenix_env"; phenix.chain_comparison "$reference" "$model" \
            max_dist=3 unique_part_of_target_only=False test_unique_part_of_target_only=False \
            use_crystal_symmetry=False) > "$out/chain-comparison.txt"
    fi
done
